# Notebook 05 — Reporting & Dashboard

NB05 is the notebook whose output a regulator would read first. Its central design choice is that it **is a structure, not a verdict**. The executive summary is a placeholder ('[ANALYST: Executive Summary — insert 3 key findings here]'), every compliance status is a placeholder ('[ANALYST: insert status]'), and the recommendations section is removed entirely. The analyst fills in the substance; the notebook produces the scaffolding.

The notebook's claim-authorization self-check at the end asserts:

- The executive summary placeholder is present.

- No forbidden tokens ("Compliant", "Non-Compliant", "PASS", "FAIL", "Adequate", "Inadequate", "Validated") appear anywhere in the report.

If either assertion fails, the notebook raises. The report is only "valid" if the assertions pass.

### Interpreting the key results

**The reconciled results (from NB03 + NB04 contracts) are the numbers a reader should anchor on**. Base PD 0.95%, LGD 44.28%, EAD $204,745, stage distribution 89.1/9.9/1.0, weighted ECL per loan $2,231.16. These come from the contracts, not from re-derivation, so they are guaranteed to match NB03 and NB04. The reconciliation assertion ('nb04['upstream_nb03_run_id'] == nb03['run_id']') is the contamination guard: if NB04 was built on a different NB03 run than the one NB05 is reading, the assertion fires before any report is generated.

**The reconciliation drift WARNING on** 'base_pd' **is expected and not an error**. The dataframe's mean 'pd_estimate' is 5.8%, while NB03's contract says 'unconditional_pd = 0.95%'. These are different by design: 'pd_estimate' is a per-loan score (mean 5.8%), 'unconditional_pd' is the sample default rate (0.95%). The notebook logs the warning but proceeds with NB03's contract. A reader seeing "Reconciliation drift" in the log should not conclude the pipeline is broken; the narrative should say so explicitly.

**The report is deliberately incomplete in the right places**. The executive summary, compliance statuses, and analyst sign-off are all placeholders. A reader who skims the report and sees blank statuses might conclude the report is unfinished — it is, and that is the point. The analyst's job is to fill them in. The narrative should say: "The blank fields are not omissions. They are the analyst's contribution, and the notebook's claim-authorization discipline is what keeps them blank."

**The decision log (D001–D025) is the interpretive skeleton for the Validation Opinion**. It consolidates 25 decisions across data quality, feature engineering, PD calibration, SICR, LGD, EAD, scenario, capital, fairness, and reporting. D021–D025 are the more recent decisions (exclude 'delinquency_state' from correlation, exclude 'risk_fico' from prediction, CCF = 0, portfolio-aggregate ECL for coverage, do not conclude fairness failure on geographic proxies). A reader who wants to understand "what choices did the analyst make and why?" reads this file. The narrative should point to it.

**The assumptions narrative is generated from the CSV, not hand-written**. The header shows both the source register's run_id and the narrative's own run_id. This is the provenance discipline — the narrative can't drift from the CSV because it's generated from it.

In [1]:
# Notebook 05 — Reporting & Dashboard
# Version 1.2  [EDIT E-N05-VER] bumped from 1.1
# Changes vs 1.1:
#   E-N05-VER    version bump
#   E-N05-01     C5 — removed `from sklearn.metrics import roc_curve, auc`
#   E-N05-02     canonical load + FileNotFoundError guard (E-G01)
#   E-N05-03     results read from NB03/NB04 JSON contracts, not re-derived (reconciliation assert vs both contracts)
#   E-N05-04     claim-authorization: executive summary = placeholder;
#                compliance table = placeholders; recommendations removed
#   E-N05-05     manifest scoped to analysis artifacts (was walking entire FS)
#   E-N05-06     run_id / notebook_version on every tabular artifact
#   E-N05-07     assumptions narrative reads run_id from CSV, not its own run
#
# NOTE: This is the fifth notebook of the Fannie Mae pipeline, and the most claim-authorization-sensitive notebook in the whole pipeline. It is
# the notebook whose output a regulator would read first.
# 
# The core tension NB05 has to resolve:
# - The report must contain the compliance assessment (that's its job). 
# - But the compliance assessment is the ANALYST'S determination, not the notebook's.
# 
# The v1.2 resolution: the report is a STRUCTURE, not a verdict. It has sections for the compliance statement, but the compliance statuses are
# [ANALYST: insert status] placeholders. Same for the executive summary — the notebook produces the scaffolding, the analyst fills in the findings.
# 
# This is the E-N05-04 fix. The v1.1 report had the notebook writing its own conclusions ("The model is compliant with..."). That was a claim
# the notebook wasn't authorized to make.
# 
# What breaks if you change this notebook without updating downstream:
# - NB06 reads the pipeline manifest (to know what to export) and the compliance report (to know what to put in the data dictionary).
#   Changing either file's schema breaks NB06.
# - Any reviewer reading this notebook's output expects the report to follow the v1.2 structure. Changing section names or the placeholder
#   format means updating the Validation Opinion template too.
# 
# What this notebook does NOT do:
# - Compute any new model metrics. All metrics come from NB03/NB04 contracts (E-N05-03).
# - Declare compliance. It produces a report structure with placeholders.
# - Recommend actions. Recommendations are the analyst's, not the notebook's.
# - Produce the executive summary. Placeholder only.
# - Compute ROC/AUC. C5 scope guard (E-N05-01).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import logging
from datetime import datetime
import os
import sys
import json
import hashlib
from pathlib import Path

warnings.filterwarnings('ignore')

# NOTE: Same version-bump discipline as NB01–NB04. NB05's version appears in the audit log header, the manifest, the assumptions narrative, the
# decision log, and the compliance report. Any change to the report structure or the placeholder format is a MINOR bump; a change to the
# manifest globs or the claim-authorization checks is a MAJOR bump (because reviewers depend on the current behavior).
NOTEBOOK_VERSION = "1.2"

# NOTE: [E-N05-02] Four canonical paths. Every input NB05 needs is declared here as a constant, and the file existence check below uses
# this list. If you add a new input, add it to both places.
# 
# [E-N05-03] The NB03_PARAMS and NB04_RESULTS paths are the contracts. NB05 reads them rather than re-deriving metrics from df. See the
# reconciliation asserts below for why this matters.
CLEAN_PATH = 'data/fannie_mae_final_clean.csv'
ASSUMPTIONS_PATH = 'data/assumptions_register.csv'
NB03_PARAMS = 'outputs/notebook_03_parameters.json'
NB04_RESULTS = 'outputs/notebook_04_results.json'

### SETUP AUDIT TRAIL

In [3]:
def compute_file_hash(path):
    """SHA-256-16 of a file, chunked read.

    >> NOTE: Same function as NB01–NB04. Used for the manifest and for individual artifact hashes in the audit log. Enables a reviewer to
    >> verify no file was modified after being saved.
    """
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]


def setup_audit_logger(run_id=None):
    """Same pattern as NB01–NB04.

    >> NOTE: Distinct log filename (reporting_audit_trail_{run_id}.log). Same structural pattern as the other notebooks. If you add a new
    >> artifact, its hash gets logged here automatically via _stamp_and_save or _write_text_with_header.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs('logs', exist_ok=True)
    logger = logging.getLogger(f"reporting_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(
        f"logs/reporting_audit_trail_{run_id}.log", encoding='utf-8')
    handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(console_handler)
    return logger, run_id


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"NOTEBOOK 05 v{NOTEBOOK_VERSION}: REPORTING & DASHBOARD")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

print(f"\n✅ Audit trail: logs/reporting_audit_trail_{RUN_ID}.log")

2026-10-22 13:51:53 | INFO | ============================================================
2026-10-22 13:51:53 | INFO | NOTEBOOK 05 v1.2: REPORTING & DASHBOARD
2026-10-22 13:51:53 | INFO | RUN ID: 20261022_135153
2026-10-22 13:51:53 | INFO | ============================================================

✅ Audit trail: logs/reporting_audit_trail_20261022_135153.log


### SECTION 1: LOAD DATA AND RESULTS

In [4]:
# NOTE: [E-N05-02] The pre-flight check. If ANY of the four upstream artifacts is missing, the notebook fails immediately — no data loading,
# no report generation. This is deliberate: a compliance report built on partial inputs is worse than no report, because it looks complete but isn't.
# 
# The four paths are:
# - CLEAN_PATH: the cleaned dataframe (used for the default rate and count) 
# - ASSUMPTIONS_PATH: NB01's assumptions register (used for the narrative)
# - NB03_PARAMS: the calibration contract (used for PD/LGD/EAD/stages)
# - NB04_RESULTS: the stress+fairness contract (used for ECL/scenarios/severity/fairness)
# 
# Missing any of these means an upstream notebook didn't run (or ran with a different version). The error message names the specific file and the
# specific version requirement.
for p in [CLEAN_PATH, ASSUMPTIONS_PATH, NB03_PARAMS, NB04_RESULTS]:
    if not Path(p).exists():
        raise FileNotFoundError(
            f"Upstream artifact missing: {p}. "
            f"Run Notebooks 01–04 (v1.2+) before Notebook 05."
        )

df = pd.read_csv(CLEAN_PATH)
assumptions_df = pd.read_csv(ASSUMPTIONS_PATH)

print(f"✅ Loaded clean data: {df.shape}, "
      f"Default Rate: {df['default'].mean():.2%}")
print(f"✅ Loaded assumptions: {len(assumptions_df)} entries")

# NOTE: [E-N05-03] The most consequential fix in NB05 v1.1 → v1.2. In v1.1, the notebook RE-DERIVED the metrics from the dataframe — computing
# its own base_pd, base_lgd, base_ead by re-running the calibration calculation. That produced values that could silently drift from NB03's if:
# - NB01 or NB02 re-ran with different parameters
# - NB03 had a bug in one computation path but not the other
# - Anyone manually edited the intermediate files
# 
# The v1.2 fix: NB05 reads NB03's contract (notebook_03_parameters.json) and NB04's contract (notebook_04_results.json) rather than re-deriving.
# The report's numbers are therefore guaranteed to match what the calibration and stress-testing notebooks produced.
# 
# The E-N05-03 fix also adds the reconciliation asserts below — NB05 verifies that NB04 was built on the same NB03 run that NB05 is reading.
# If they diverge, the assertion fires before any report is generated.
with open(NB03_PARAMS) as f:
    nb03 = json.load(f)
with open(NB04_RESULTS) as f:
    nb04 = json.load(f)

# NOTE: The base values, extracted from the contracts. These are the authoritative values for the report. If you're tempted to compute them
# from df again, don't — the whole point of E-N05-03 is that the report must reflect what the upstream notebooks produced.
base_pd = nb03['results']['unconditional_pd']
base_lgd = nb03['results']['average_lgd']
base_ead = nb03['results']['average_ead']
stage_distribution = nb03['results']['stage_distribution']
total_weighted = nb04['total_weighted_ecl']
scenario_results = nb04['scenarios']

audit_logger.info(
    f"Loaded NB03 contract run_id={nb03['run_id']}; "
    f"NB04 contract run_id={nb04['run_id']} (upstream_nb03_run_id={nb04['upstream_nb03_run_id']})"
)
# NOTE: The reconciliation assertion — the most important guardrail in this notebook. NB04's results JSON records `upstream_nb03_run_id`,
# which is the run_id of the NB03 that produced NB04's inputs. NB05 asserts that this matches the run_id of the NB03 whose parameters NB05
# is reading.
# 
# If they diverge, the report would be mixing parameters from one NB03 run with results from another. The numbers would look plausible but
# wouldn't correspond to any coherent model. The assertion catches this before the report is generated.
# 
# Common cause of the assertion firing: NB03 re-ran after NB04 ran. Fix: re-run NB04 (and NB05) so all three notebooks use the same NB03 run.
assert nb04['upstream_nb03_run_id'] == nb03['run_id'], (
    f"Contamination risk: NB04 built on NB03 run {nb04['upstream_nb03_run_id']} "
    f"but NB05 is reading NB03 run {nb03['run_id']}."
)

# NOTE: The second reconciliation — NB05 checks that the dataframe's mean pd_estimate matches what NB03's contract says. If they differ, log a
# WARNING but proceed with NB03's contract value. Same pattern as E-N04-02: prefer the upstream contract, but surface the divergence.
# 
# Divergence can happen legitimately if:
# - NB03 applied clipping that the raw mean doesn't
# - The dataframe was regenerated but NB03's contract is from the old run
# 
# The WARNING is not a failure. It's a signal to the analyst that the two numbers are different — which might matter if they're investigating something.
if 'pd_estimate' in df.columns:
    local_pd = df['pd_estimate'].mean()
    if abs(local_pd - base_pd) > 1e-6:
        audit_logger.warning(
            f"Reconciliation drift on base_pd: dataframe={local_pd:.6f}, "
            f"NB03 contract={base_pd:.6f}. Using NB03 contract."
        )

print("\n" + "-" * 60)
print("RECONCILED RESULTS (from NB03 + NB04 contracts)")
print("-" * 60)
print(f"Base PD:  {base_pd:.2%}")
print(f"Base LGD: {base_lgd:.2%}")
print(f"Base EAD: ${base_ead:,.0f}")
print(f"Stage Distribution:")
for stage, pct in stage_distribution.items():
    print(f"  {stage}: {pct:.1%}")
print(f"Probability-Weighted ECL (per loan): ${total_weighted:,.2f}")

# NOTE: [E-N05-06] Same save helpers as NB01–NB04. _stamp_and_save for CSVs (adds run_id and notebook_version columns), _write_text_with_header
# for text files (adds a run_id and version header). Every artifact NB05 writes goes through one of these. If you add a new artifact, use one of
# these helpers — do not write files directly.
def _stamp_and_save(dataframe, path):
    """Save a DataFrame with run_id / notebook_version columns."""
    out = dataframe.copy()
    out['run_id'] = RUN_ID
    out['notebook_version'] = NOTEBOOK_VERSION
    out.to_csv(path, index=False)
    audit_logger.info(f"Saved: {path} hash={compute_file_hash(path)}")


def _write_text_with_header(lines, path):
    """Write a text file with a run_id / version header."""
    header = [
        "=" * 80,
        f"Run ID: {RUN_ID}",
        f"Notebook Version: {NOTEBOOK_VERSION}",
        "=" * 80,
    ]
    with open(path, 'w', encoding='utf-8') as f:
        f.write("\n".join(header + lines))
    audit_logger.info(f"Saved: {path} hash={compute_file_hash(path)}")

✅ Loaded clean data: (100000, 175), Default Rate: 0.95%
✅ Loaded assumptions: 10 entries
2026-10-22 13:51:54 | INFO | Loaded NB03 contract run_id=20261022_133307; NB04 contract run_id=20261022_134434 (upstream_nb03_run_id=20261022_133307)
2026-10-22 13:51:54 | WARNING | Reconciliation drift on base_pd: dataframe=0.058076, NB03 contract=0.009530. Using NB03 contract.

------------------------------------------------------------
RECONCILED RESULTS (from NB03 + NB04 contracts)
------------------------------------------------------------
Base PD:  0.95%
Base LGD: 44.28%
Base EAD: $204,745
Stage Distribution:
  Stage 1: 89.1%
  Stage 2: 9.9%
  Stage 3: 1.0%
Probability-Weighted ECL (per loan): $2,231.16


### SECTION 2: NARRATIVE ASSUMPTIONS REGISTER

In [5]:
# NOTE: [E-N05-07] The narrative is generated FROM the assumptions CSV, not hand-written. This ensures the narrative can't drift from the CSV — any change to the CSV automatically propagates to the narrative.
# The run_id issue was the E-N05-07 fix. In v1.1, the narrative header showed NB05's own run_id (the run that generated the narrative), which
# was misleading — it implied the assumptions were frozen in that run.
# But the assumptions were frozen in a DIFFERENT run (NB01's run). The v1.2 fix reads the run_id from the CSV itself (which was stamped by
# NB01 via E-N01-01), so the header shows the correct source run.
# 
# The header now has TWO run_ids:
# - "Source register run_id": the NB01 run that produced the assumptions 
# - "This narrative generated by run_id": the NB05 run that produced the narrative
# These may be the same (if NB01-05 all ran in one go) or different (if NB01 ran earlier). The distinction matters for lineage.
_nb01_run_ids = (
    assumptions_df['run_id'].dropna().unique().tolist()
    if 'run_id' in assumptions_df.columns else []
)
_source_run = _nb01_run_ids[0] if _nb01_run_ids else "not_recorded_in_csv"

narrative_lines = [
    "",
    "ASSUMPTIONS REGISTER — NARRATIVE VERSION",
    "",
    f"Source register run_id: {_source_run}",
    f"This narrative generated by run_id: {RUN_ID}",
    "",
    "This document explains each assumption in the register. For the",
    "machine-readable version, see data/assumptions_register.csv.",
    "",
]

for _, row in assumptions_df.iterrows():
    narrative_lines.append(f"## {row['Parameter']}")
    narrative_lines.append(f"**Value:** {row['Value']}")
    narrative_lines.append(f"**Rationale:** {row['Rationale']}")
    narrative_lines.append(f"**Regulatory Reference:** {row['Regulatory_Reference']}")
    narrative_lines.append(f"**Used in:** Notebook {row['Notebook']}, {row['Section']}")
    narrative_lines.append("")

_write_text_with_header(narrative_lines,
                        'outputs/assumptions_register_narrative.md')
print("✅ Saved: outputs/assumptions_register_narrative.md")

2026-10-22 13:51:54 | INFO | Saved: outputs/assumptions_register_narrative.md hash=3e905d6c45bb931c
✅ Saved: outputs/assumptions_register_narrative.md


### SECTION 3: DECISION LOG

In [6]:
# NOTE: The decision log consolidates 25 decisions (D001-D025) from across the pipeline. It's a single reference artifact for the
# Validation Opinion to cite.
# 
# Each decision has:
# - id: a stable identifier (D001-D025)
# - category: what kind of decision (Data Quality, Feature Engineering, etc.)
# - decision: what was decided
# - rationale: why
# - reg_ref: which regulatory reference applies
# 
# The decisions are ordered by ID, which roughly follows the pipeline order (D001-D005 are data quality, D006-D009 are SICR, etc.). D021-D025
# are the newer decisions added during the v1.2 revisions.
# 
# If you add a decision, use the next available ID. If you change one, don't renumber — a change to D007 is a change to D007, not a replacement
# with a new ID. Renumbering breaks the Validation Opinion's cross-references.

decisions = [
    {'id': 'D001', 'category': 'Data Quality', 'decision': 'Random sampling (100,000 rows)',
     'rationale': 'Prevents temporal bias; reproducible', 'reg_ref': 'OSFI B-13 s.3.2'},
    {'id': 'D002', 'category': 'Data Quality', 'decision': 'Median imputation for numeric',
     'rationale': 'Robust to outliers', 'reg_ref': 'OSFI E-23 s.4.3'},
    {'id': 'D003', 'category': 'Feature Engineering', 'decision': 'LTV threshold: 80%',
     'rationale': 'Aligns with PMI/CMHC', 'reg_ref': 'HMDA, PMI'},
    {'id': 'D004', 'category': 'Feature Engineering', 'decision': 'FICO bands: 680, 720, 760',
     'rationale': 'Industry standard', 'reg_ref': 'Fannie Mae risk bands'},
    {'id': 'D005', 'category': 'PD Calibration', 'decision': '12-month observation window',
     'rationale': 'IFRS 9 requirement', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D006', 'category': 'PD Calibration', 'decision': 'PIT PD',
     'rationale': 'IFRS 9 requires current conditions', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D007', 'category': 'SICR', 'decision': 'SICR trigger: PD > 200%',
     'rationale': 'Industry standard', 'reg_ref': 'IFRS 9 s.5.5.9'},
    {'id': 'D008', 'category': 'SICR', 'decision': 'SICR trigger: FICO drop > 50',
     'rationale': 'Material credit decline', 'reg_ref': 'IFRS 9 s.5.5.9'},
    {'id': 'D009', 'category': 'SICR', 'decision': '30 DPD backstop',
     'rationale': 'IFRS 9 requirement', 'reg_ref': 'IFRS 9 s.5.5.11'},
    {'id': 'D010', 'category': 'LGD', 'decision': 'Base LGD: 45%',
     'rationale': 'Historical recovery', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D011', 'category': 'LGD', 'decision': 'LTV-based LGD adjustments',
     'rationale': 'Collateral cushion effect', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D012', 'category': 'EAD', 'decision': 'Prepayment rate: 8%',
     'rationale': 'Historical average', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D013', 'category': 'EAD', 'decision': 'Curtailment rate: 2%',
     'rationale': 'Historical average', 'reg_ref': 'IFRS 9 s.5.5.3'},
    {'id': 'D014', 'category': 'Scenario', 'decision': 'Four scenarios',
     'rationale': 'IFRS 9 + tail risk', 'reg_ref': 'IFRS 9 s.5.5.18'},
    {'id': 'D015', 'category': 'Scenario', 'decision': 'Weights: 45/30/15/10',
     'rationale': 'Probability-weighted ECL', 'reg_ref': 'IFRS 9 s.5.5.17'},
    {'id': 'D016', 'category': 'Scenario', 'decision': 'Tail multipliers: 4.5x PD, 2.0x LGD',
     'rationale': '2008 crisis × 1.5', 'reg_ref': 'IFRS 9 s.5.5.18'},
    {'id': 'D017', 'category': 'Capital', 'decision': 'CET1: 4.5%',
     'rationale': 'Basel III minimum', 'reg_ref': 'Basel III'},
    {'id': 'D018', 'category': 'Fairness', 'decision': 'Protected attributes: geographic proxies',
     'rationale': 'No protected attributes available', 'reg_ref': 'ECOA, OSFI E-23 s.6.0'},
    {'id': 'D019', 'category': 'Fairness', 'decision': '4/5ths rule (80%)',
     'rationale': 'ECOA standard', 'reg_ref': 'ECOA'},
    {'id': 'D020', 'category': 'Reporting', 'decision': 'Static dashboard',
     'rationale': 'Regulatory submission', 'reg_ref': 'OSFI E-23 s.4.2'},
    {'id': 'D021', 'category': 'EDA', 'decision': 'Exclude delinquency_state from correlation',
     'rationale': 'It is a target transform, not an independent feature',
     'reg_ref': 'SR 11-7 §V'},
    {'id': 'D022', 'category': 'EDA', 'decision': 'Exclude risk_fico from prediction',
     'rationale': 'Monotone transform of FICO_BOR; redundant', 'reg_ref': 'SR 11-7 §V'},
    {'id': 'D023', 'category': 'EAD', 'decision': 'CCF = 0 for term mortgages',
     'rationale': 'Basel III CCF applies only to revolving credit',
     'reg_ref': 'Basel III (CCF scope)'},
    {'id': 'D024', 'category': 'Capital', 'decision': 'Portfolio-aggregate ECL for coverage diagnostic',
     'rationale': 'Per-loan ECL vs. portfolio capital is a scale mismatch',
     'reg_ref': 'Basel III'},
    {'id': 'D025', 'category': 'Fairness',
     'decision': 'Do not conclude failure on geographic proxies',
     'rationale': '4/5ths rule is misapplied to high-cardinality splits',
     'reg_ref': 'ECOA, OSFI E-23 s.6.0'},
]

log_lines = ["", "DECISION LOG", ""]
for d in decisions:
    log_lines.append("=" * 60)
    log_lines.append(f"DECISION: {d['id']} - {d['category']}")
    log_lines.append("=" * 60)
    log_lines.append(f"Decision: {d['decision']}")
    log_lines.append(f"Rationale: {d['rationale']}")
    log_lines.append(f"Regulatory Reference: {d['reg_ref']}")
    log_lines.append("")

_write_text_with_header(log_lines, 'outputs/decision_log_v1_2.txt')
print(f"✅ Saved: outputs/decision_log_v1_2.txt ({len(decisions)} decisions)")


2026-10-22 13:51:54 | INFO | Saved: outputs/decision_log_v1_2.txt hash=fb4e4e613ba2b2ad
✅ Saved: outputs/decision_log_v1_2.txt (25 decisions)


### SECTION 4: PIPELINE MANIFEST

In [7]:
# NOTE: [E-N05-05] The manifest scope fix. In v1.1, the manifest walked the entire filesystem with `os.walk('.')`, which produced a manifest
# with tens of thousands of files — most of them irrelevant to the analysis. A reviewer couldn't tell which files mattered.
# 
# The v1.2 fix: a fixed list of glob patterns (MANIFEST_GLOBS) that captures only analysis artifacts. The manifest now has ~100-150 entries,
# all relevant.
# 
# If you add a new artifact directory, add its glob pattern here. If you find yourself wanting to walk the whole filesystem again, don't — the
# manifest is a record of analysis artifacts, not a filesystem snapshot.
MANIFEST_GLOBS = [
    'data/assumptions_register.csv',
    'data/fannie_mae_raw_sample.csv',
    'data/fannie_mae_final_clean.csv',
    'outputs/*.csv',
    'outputs/*.txt',
    'outputs/*.md',
    'outputs/*.png',
    'outputs/*.json',
    'powerbi_exports/*.csv',
    'powerbi_exports/*.md',
]
import glob
paths = []
for pat in MANIFEST_GLOBS:
    paths.extend(glob.glob(pat))

# NOTE: Two filters applied to the path list:
# 1. `'manifest' not in p` — exclude any file with "manifest" in the name. This prevents the manifest from listing itself, which would
#     be a chicken-and-egg problem (the manifest's hash depends on its own contents).
# 2. `p.endswith((...))` — only keep analysis artifact extensions. This catches stray files that got written to outputs/ by accident.
paths = [
    p for p in sorted(set(paths))
    if 'manifest' not in p and p.endswith(('.csv', '.txt', '.md', '.png', '.json'))
]

manifest_rows = []
for p in paths:
    try:
        manifest_rows.append({
            'file': p,
            'hash_sha256_16': compute_file_hash(p),
            'size_bytes': os.path.getsize(p),
            'run_id': RUN_ID,
            'notebook_version': NOTEBOOK_VERSION,
            'timestamp': datetime.now().isoformat(),
        })
    except Exception as e:
        # NOTE: If a file can't be read (e.g., locked by another process, corrupted, deleted between glob and read), the loop logs a
        # WARNING and continues. It doesn't crash the whole notebook.
        # This is deliberate: the manifest is a record, not a critical computation. A missing file shouldn't block the report.
        audit_logger.warning(f"Manifest skip {p}: {e}")

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv('outputs/manifest_pipeline_v1_2.csv', index=False)
print(f"✅ Saved: outputs/manifest_pipeline_v1_2.csv ({len(manifest_df)} files)")
audit_logger.info(f"Manifest: {len(manifest_df)} files hashed")

✅ Saved: outputs/manifest_pipeline_v1_2.csv (126 files)
2026-10-22 13:51:55 | INFO | Manifest: 126 files hashed


### SECTION 5: COMPLIANCE REPORT

In [8]:
# NOTE: [E-N05-04] The most consequential fix in NB05 v1.1 → v1.2. In v1.1, the compliance report had the notebook writing its own executive
# summary and its own compliance status. That was a claim the notebook wasn't authorized to make.
# 
# The v1.2 fix: the executive summary is a PLACEHOLDER. Every compliance status is a PLACEHOLDER. The recommendations section is REMOVED (not
# "moved elsewhere" — removed entirely, because recommendations belong in the Validation Opinion, not the compliance report).
# 
# The placeholders are formatted as [ANALYST: <description>] — the same format used in every other notebook. A reviewer scanning the report sees
# the placeholders and knows exactly where the analyst needs to fill in.
# 
# This is the single most important structural choice in NB05. The report is a SCAFFOLD, not a verdict. The analyst fills in the verdict.

report_lines = [
    "=" * 80,
    f"IFRS 9 COMPLIANCE REPORT (v{NOTEBOOK_VERSION})",
    f"Run ID: {RUN_ID}",
    "=" * 80,
    f"\nReport Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"NB03 contract run_id: {nb03['run_id']}",
    f"NB04 contract run_id: {nb04['run_id']}",

    "\n" + "-" * 60,
    "1. EXECUTIVE SUMMARY",
    "-" * 60,
    # NOTE: The executive summary placeholder is the most visible claim-authorization marker in the whole report. A reviewer who
    # reads the executive summary sees "[ANALYST: Executive Summary — insert 3 key findings here]" and knows the analyst needs to fill
    # it in. If the notebook auto-generated the summary, the reviewer would be reading the notebook's findings, not the analyst's.
    "[ANALYST: Executive Summary — insert 3 key findings here. Each finding",
    " must be authorized by the analyst and must not be generated by the tool.]",

    "\n" + "-" * 60,
    "2. REGULATORY COMPLIANCE SUMMARY",
    "-" * 60,
    # NOTE: The compliance table is a TEMPLATE. Each row has a regulation, a requirement, and an evidence artifact — but the compliance status
    # is [ANALYST: insert status]. The notebook provides the table structure; the analyst fills in the statuses.
    # 
    # The evidence artifact column is important: it points to the specific file that demonstrates the requirement is met. A reviewer can
    # click through to the artifact. The status column is where the analyst declares whether the evidence is sufficient.
    "| Regulation | Requirement | Evidence artifact | Compliance status |",
    "|---|---|---|---|",
    f"| IFRS 9 s.5.5.3 | PD/LGD/EAD calibration | outputs/pd_lgd_ead_calibration_summary_v1_3.txt | [ANALYST: insert status] |",
    f"| IFRS 9 s.5.5.9 | SICR assessment | Notebook 03 stage output | [ANALYST: insert status] |",
    f"| IFRS 9 s.5.5.11 | 30-DPD backstop | Notebook 03 stage output | [ANALYST: insert status] |",
    f"| IFRS 9 s.5.5.17 | Probability-weighted ECL | outputs/notebook_04_results.json | [ANALYST: insert status] |",
    f"| IFRS 9 s.5.5.18 | Reasonable & supportable scenarios | Notebook 04 scenario definitions | [ANALYST: insert status] |",
    f"| OSFI E-23 s.4.2 | Documentation | This report + decision log | [ANALYST: insert status] |",
    f"| OSFI E-23 s.6.0 | Fairness testing | outputs/notebook_04_fairness_limitations.csv | [ANALYST: insert status] |",
    f"| Basel III | CCF scope (term mortgages) | Notebook 03 EAD output (CCF=0) | [ANALYST: insert status] |",

    "\n" + "-" * 60,
    "3. KEY RESULTS (numbers only — no interpretation)",
    "-" * 60,
    # NOTE: Section 3 is the ONE part of the report that's fully auto-generated, because it's purely numbers. No claims, no
    # interpretations, no statuses. Just the values.
    # 
    # The header explicitly says "(numbers only — no interpretation)" so a reader knows this section is factual, not analytical. If the
    # notebook ever adds a sentence to this section, that's a violation of E-N05-04.
    f"Default rate:             {df['default'].mean():.2%}",
    f"Unconditional PD:         {base_pd:.2%}",
    f"Average LGD:              {base_lgd:.2%}",
    f"Average EAD:              ${base_ead:,.0f}",
    f"Stage 1 / 2 / 3:          {stage_distribution.get('Stage 1', 0):.1%} / "
    f"{stage_distribution.get('Stage 2', 0):.1%} / "
    f"{stage_distribution.get('Stage 3', 0):.1%}",
    f"Weighted ECL (per loan):  ${total_weighted:,.2f}",
    f"ECL ratio of EAD:         {nb04['ecl_ratio_of_ead']:.3%}",
    f"PD-cap effect (Tail):     {nb04['pd_cap_sensitivity']['cap_effect_pct']:.1f}% understated",
    f"Severity check ratio:     {nb04['severity_check']['ratio_vs_historical']:.2f}x historical",

    "\n" + "-" * 60,
    "4. KNOWN LIMITATIONS (list only — no interpretation)",
    "-" * 60,
    # >> NOTE: The limitations list is factual — each limitation is a statement of scope. The notebook lists them; the analyst decides
    # >> whether they're acceptable. No interpretation, no recommendations on mitigation (that's in the Remediation Plan).
    "L1  PD-LGD correlation is constructed (not empirical).",
    "L2  SICR uses top-decile proxy (not true SICR).",
    f"L3  LGD is a function of PD; average {base_lgd:.2%} outside 40–60% benchmark.",
    "L4  Prepayment/curtailment are assumptions (8% / 2%).",
    "L5  EIR is industry average (3.5%).",
    f"L6  PD cap ({nb04['config']['pd_cap']:.0%}) masks tail convexity; "
    f"Tail ECL understated ~{nb04['pd_cap_sensitivity']['cap_effect_pct']:.1f}%.",
    "L7  Fairness testing incomplete — no protected attributes; cardinality >50.",
    "L8  US data used as proxy for Canadian portfolio.",
    "L9  Scenario weights and multipliers are assumption-based (not calibrated to history).",
    "L10 Capital coverage metric is a diagnostic; CET1 is a ratio of RWA, not total assets.",

    "\n" + "-" * 60,
    "5. ANALYST SIGN-OFF",
    "-" * 60,
    # NOTE: The sign-off block is intentionally blank. The analyst fills in name, role, date, signature. The notebook provides the space;
    # the analyst provides the signature.
    "Name: ____________________   Role: ____________________",
    "Date: ____________________   Signature: ____________________",
    "",
]

with open('outputs/ifrs9_compliance_report_v1_2.txt', 'w', encoding='utf-8') as f:
    f.write("\n".join(report_lines))
print("✅ Saved: outputs/ifrs9_compliance_report_v1_2.txt")
audit_logger.info(
    f"Saved: outputs/ifrs9_compliance_report_v1_2.txt "
    f"hash={compute_file_hash('outputs/ifrs9_compliance_report_v1_2.txt')}"
)

✅ Saved: outputs/ifrs9_compliance_report_v1_2.txt
2026-10-22 13:51:55 | INFO | Saved: outputs/ifrs9_compliance_report_v1_2.txt hash=c18f540a223d3e05


### SECTION 6: CLAIM-AUTHORIZATION SELF-CHECK

In [9]:
# NOTE: [E-N05-04] The self-check. This is the guardrail that ensures the claim-authorization discipline actually held. The notebook asserts:
# 
# 1. The executive summary is a placeholder (contains the literal string "[ANALYST: Executive Summary")
# 2. No forbidden tokens appear in the report ("Compliant", "Non-Compliant", "PASS", "FAIL", "Adequate", "Inadequate", "Validated")
# 
# If either assertion fails, the notebook raises an error and the report is NOT considered valid. This is the enforcement mechanism — the report
# is only valid if the assertions pass.
# 
# The assertions fire on every run. If you remove them, you're removing the guardrail that prevents the tool from writing its own verdict. Don't.
report_text = open('outputs/ifrs9_compliance_report_v1_2.txt',
                    encoding='utf-8').read()

# The executive summary section must contain the placeholder.
assert "[ANALYST: Executive Summary" in report_text, (
    "Claim-authorization violation: executive summary must be a placeholder."
)

# Every compliance status must be a placeholder — no "Compliant"/"Non-Compliant". NOTE: The token list is deliberately comprehensive. It's not just
# "Compliant" and "Non-Compliant" — it's also "PASS"/"FAIL" (which could slip in via a status column), "Adequate"/"Inadequate" (which could slip
# in via the capital coverage section), and "Validated" (which could slip in via the severity check section).
# 
# If you add a new forbidden token, add it here. The point is to catch every variant of "the tool wrote its own verdict".
for token in ["Compliant", "Non-Compliant", "PASS", "FAIL",
              "Adequate", "Inadequate", "Validated"]:
    assert token not in report_text, (
        f"Claim-authorization violation: '{token}' token found in report."
    )

audit_logger.info("Claim-authorization self-check: PASS")

2026-10-22 13:51:55 | INFO | Claim-authorization self-check: PASS


### COMPLETE

In [10]:
print(f"\n{'=' * 80}")
print(f"NOTEBOOK 05 v{NOTEBOOK_VERSION} COMPLETE")
print(f"{'=' * 80}")
print("""
Artifacts written:
  txt:  outputs/assumptions_register_narrative.md
        outputs/decision_log_v1_2.txt
        outputs/ifrs9_compliance_report_v1_2.txt
  csv:  outputs/manifest_pipeline_v1_2.csv
  log:  logs/reporting_audit_trail_<run_id>.log

Claim-authorization self-check: PASS
  - Executive summary is a placeholder
  - Every compliance status is a placeholder
  - No "Compliant"/"PASS"/"Adequate"/"Validated" tokens present
""")
audit_logger.info(f"NOTEBOOK 05 v{NOTEBOOK_VERSION} COMPLETE")


NOTEBOOK 05 v1.2 COMPLETE

Artifacts written:
  txt:  outputs/assumptions_register_narrative.md
        outputs/decision_log_v1_2.txt
        outputs/ifrs9_compliance_report_v1_2.txt
  csv:  outputs/manifest_pipeline_v1_2.csv
  log:  logs/reporting_audit_trail_<run_id>.log

Claim-authorization self-check: PASS
  - Executive summary is a placeholder
  - Every compliance status is a placeholder
  - No "Compliant"/"PASS"/"Adequate"/"Validated" tokens present

2026-10-22 13:51:55 | INFO | NOTEBOOK 05 v1.2 COMPLETE


## What this notebook establishes, and what it hands off

This notebook produces five artifacts:

1. `outputs/ifrs9_compliance_dashboard_enhanced.png` — the 11-panel dashboard
2. `outputs/ifrs9_compliance_report_enhanced.txt` — the written compliance report
3. `outputs/decision_log_enhanced.txt` — the 20-decision log
4. `outputs/regulatory_mapping_enhanced.txt` — the regulatory mapping matrix
5. `outputs/final_summary_enhanced.txt` — the closing summary

Plus an audit trail at `logs/reporting_audit_trail_{RUN_ID}.log`.

**What the notebook should be, and what it currently is:**

The notebook should be the translation layer — taking the results from
Notebooks 01 through 04 and reframing them as documents a decision-maker can
act on. That is what a reporting notebook is for.

The current version partially achieves this. The dashboard's data-derived
panels (default distribution, FICO bands, property-type LGDs, vintage
patterns, correlation matrix, ROC curve) accurately reflect the portfolio.
The decision log and regulatory mapping are structurally sound.

But three areas — the compliance report, the stage distribution panel, and
the fairness panel — use hardcoded values that do not match the actual
outputs of the previous notebooks. In particular, the fairness results
reported here (all attributes passing) contradict the fairness testing in
Notebook 04 (which found no valid protected attributes to test).

**What this means for the project as a whole:**

For a governance artifact, the reporting layer must reflect what the analysis
actually produced. A report that shows results the analysis did not find is
not a report — it is a claim. And a claim that contradicts the underlying
analysis is worse than no claim at all, because it undermines trust in every
artifact that depends on it.

**The single most valuable fix for this notebook:**

Load the actual results from Notebooks 03 and 04. Where a result exists, use
it. Where it does not, report "not available" and explain why. The report
will then be accurate — including about the things that did not go as planned.

**What the notebook demonstrates, when corrected:**

A reporting notebook that accurately reflects the analysis — including
limitations, unresolved findings, and adverse results — is a genuine
governance artifact. It is one a regulator can trust, because it does not
claim more than the underlying work supports.

The current version is close in structure. The remaining work is making the
content match what actually happened.